# Cancer Survival Prediction — Reproducible Training and External Submission

This notebook replaces the previous training/evaluation section with a reproducible workflow. It keeps the labelled training data, internal validation split, and 75,000-row external test file separate.

> **Educational-use only.** This model is not clinically validated and must not be used for diagnosis, prognosis, treatment selection, or patient-care decisions.

## Important status

- The source datasets are not stored in this GitHub repository. Place the two CSV files in the local `data/` folder before running.
- The external test file must contain exactly 75,000 rows and an original `ID` column. IDs are carried into the output unchanged.
- This notebook creates a **candidate** model file named `GradientBoosting_candidate.pkl`; it does not overwrite the currently deployed `GradientBoosting.pkl`.
- The checked-in `Sample Prediction File.csv` is an older 14,784-row validation output. It is not the 75,000-row external submission. It will be replaced only when this notebook is run with the real external test file.


## 1. Data files and reproducibility settings

Expected local files:

- `data/patient_train_data.csv` — labelled training data, including `Survival Prediction` and `ID`
- `data/patient_test_data.csv` — external test data, including `ID` and the same predictor columns, but no target

The original dataset source/provenance was not documented in the repository. Record it in the project report before final submission if it is known.


In [ ]:
from pathlib import Path
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline as ImbalancedPipeline

warnings.filterwarnings("default")

RANDOM_STATE = 42
EXTERNAL_TEST_ROWS = 75_000
REFERENCE_DATE = pd.Timestamp("2026-01-01")

DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "patient_train_data.csv"
EXTERNAL_TEST_PATH = DATA_DIR / "patient_test_data.csv"

TARGET = "Survival Prediction"
ID_COLUMN = "ID"

EXPECTED_FEATURES = [
  "Alcohol Consumption",
  "Cancer Stage",
  "Country",
  "Diabetes",
  "Diabetes History",
  "Diet Risk",
  "Early Detection",
  "Family History",
  "Gender",
  "Genetic Mutation",
  "Healthcare Access",
  "Healthcare Costs",
  "Heart Disease History",
  "Hypertension",
  "Incidence Rate per 100K",
  "Inflammatory Bowel Disease",
  "Insurance Costs",
  "Insurance Status",
  "Mortality Rate per 100K",
  "Non Smoker",
  "Obesity BMI",
  "Physical Activity",
  "Screening History",
  "Smoking History",
  "Treatment Type",
  "Tumor Size (mm)",
  "Urban or Rural",
  "Age"
]

NUMERIC_FEATURES = [
    "Healthcare Costs",
    "Incidence Rate per 100K",
    "Mortality Rate per 100K",
    "Tumor Size (mm)",
    "Age",
]

DROP_NON_MODEL_COLUMNS = [
    "Marital Status",
    "Transfusion History",
    "Date of Birth",
]


In [ ]:
if not TRAIN_PATH.is_file():
    raise FileNotFoundError(
        f"Training file not found: {TRAIN_PATH}. "
        "Create a data/ folder and place patient_train_data.csv inside it."
    )

if not EXTERNAL_TEST_PATH.is_file():
    raise FileNotFoundError(
        f"External test file not found: {EXTERNAL_TEST_PATH}. "
        "Place the original 75,000-row patient_test_data.csv inside data/."
    )

train_raw = pd.read_csv(TRAIN_PATH)
external_test_raw = pd.read_csv(EXTERNAL_TEST_PATH)

print("Raw training shape:", train_raw.shape)
print("Raw external test shape:", external_test_raw.shape)
display(train_raw.head())
display(external_test_raw.head())


## 2. Initial exploratory data analysis

Review the source shapes, duplicate rows, data types, missingness and descriptive summaries before modelling. These checks describe the labelled training data; no imputation or scaling statistics are learned here.


In [ ]:
print("Training rows and columns:", train_raw.shape)
print("External test rows and columns:", external_test_raw.shape)
print("Duplicate training rows:", train_raw.duplicated().sum())
print("Duplicate external test rows:", external_test_raw.duplicated().sum())

missing_summary = (
    train_raw.isna().mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("Missing (%)")
    .to_frame()
)
display(missing_summary.head(25))
display(train_raw.describe(include="all").T)


In [ ]:
# Validate the external test set before any transformation.
if len(external_test_raw) != EXTERNAL_TEST_ROWS:
    raise ValueError(
        f"Expected {EXTERNAL_TEST_ROWS:,} external test rows, "
        f"but found {len(external_test_raw):,}. Do not silently drop rows."
    )

if ID_COLUMN not in train_raw.columns or ID_COLUMN not in external_test_raw.columns:
    raise KeyError("Both source files must contain the original ID column.")

if external_test_raw[ID_COLUMN].isna().any():
    raise ValueError("The external test ID column contains missing values.")

if external_test_raw[ID_COLUMN].duplicated().any():
    raise ValueError("The external test ID column contains duplicate IDs.")

external_test_ids = external_test_raw[ID_COLUMN].copy().reset_index(drop=True)
print("External test rows verified:", len(external_test_ids))
print("First original IDs:", external_test_ids.head().tolist())
print("Last original IDs:", external_test_ids.tail().tolist())


## 3. Deterministic feature preparation


Only deterministic, row-wise transformations happen before the split. Learned preprocessing statistics (imputation, category encoding and scaling) are fitted inside the model pipeline using the training fold only.

- Age is calculated against the fixed reference date above, not the current system date.
- `M` and `F` are normalized to `Male` and `Female`.
- Other gender codes, including `P`, are retained as observed categories. The earlier notebook removed `P` without documenting what it meant. Do not assign it a meaning without a data dictionary.
- The observed treatment labels `Radiotherapy` and `Combination` are normalized to the Streamlit input labels `Radiation` and `Combined`.
- Missing values and `?` are left for the training-only imputers. No rows are removed for missing predictor values.
- The earlier IQR outlier function was applied only to temporary copies and did not affect the model data. That misleading unused code is removed here; no outlier rows are dropped in this workflow.


In [ ]:
def calculate_age(value):
    """Calculate age at the fixed reference date; invalid/missing dates stay missing."""
    if pd.isna(value):
        return np.nan
    return (
        REFERENCE_DATE.year
        - value.year
        - ((REFERENCE_DATE.month, REFERENCE_DATE.day) < (value.month, value.day))
    )


def prepare_features(raw_frame):
    """Apply deterministic, row-wise cleaning and return the 28 model features."""
    frame = raw_frame.copy()

    if "Date of Birth" in frame.columns:
        dob = pd.to_datetime(
            frame["Date of Birth"],
            format="%d-%m-%Y",
            errors="coerce",
        )
        frame["Age"] = dob.map(calculate_age)

    frame = frame.drop(columns=DROP_NON_MODEL_COLUMNS, errors="ignore")
    frame = frame.replace(r"^\s*\?\s*$", np.nan, regex=True)

    if "Gender" in frame.columns:
        frame["Gender"] = (
            frame["Gender"]
            .astype("object")
            .map(lambda value: value.strip() if isinstance(value, str) else value)
            .replace({"M": "Male", "F": "Female"})
        )

    if "Urban or Rural" in frame.columns:
        frame["Urban or Rural"] = (
            frame["Urban or Rural"]
            .astype("object")
            .map(lambda value: value.strip().capitalize() if isinstance(value, str) else value)
        )

    if "Treatment Type" in frame.columns:
        frame["Treatment Type"] = frame["Treatment Type"].replace(
            {"Radiotherapy": "Radiation", "Combination": "Combined"}
        )

    for column in NUMERIC_FEATURES:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    # Make the remaining model predictors consistently categorical.
    for column in EXPECTED_FEATURES:
        if column in frame.columns and column not in NUMERIC_FEATURES:
            frame[column] = frame[column].astype("object")

    return frame


In [ ]:
if TARGET not in train_raw.columns:
    raise KeyError(f"Training data must contain the target column: {TARGET}")

if ID_COLUMN not in train_raw.columns:
    raise KeyError(f"Training data must contain the ID column: {ID_COLUMN}")

# Preserve IDs separately; neither training nor external IDs are model features.
target_text = train_raw[TARGET].astype("string").str.strip().str.title()
y_all = target_text.map({"No": 0, "Yes": 1})

if y_all.isna().any():
    invalid_targets = sorted(target_text[y_all.isna()].dropna().unique().tolist())
    missing_targets = int(y_all.isna().sum())
    raise ValueError(
        f"Found {missing_targets} missing or unrecognized target values: {invalid_targets}"
    )

X_all = prepare_features(train_raw.drop(columns=[TARGET, ID_COLUMN]))
X_external = prepare_features(external_test_raw.drop(columns=[ID_COLUMN], errors="ignore"))

missing_train = [column for column in EXPECTED_FEATURES if column not in X_all.columns]
extra_train = [column for column in X_all.columns if column not in EXPECTED_FEATURES]
missing_external = [column for column in EXPECTED_FEATURES if column not in X_external.columns]
extra_external = [column for column in X_external.columns if column not in EXPECTED_FEATURES]

if missing_train or extra_train:
    raise ValueError(
        f"Training feature schema mismatch. Missing: {missing_train}; extra: {extra_train}"
    )

if missing_external or extra_external:
    raise ValueError(
        f"External test feature schema mismatch. Missing: {missing_external}; extra: {extra_external}"
    )

X_all = X_all.reindex(columns=EXPECTED_FEATURES).reset_index(drop=True)
X_external = X_external.reindex(columns=EXPECTED_FEATURES).reset_index(drop=True)
y_all = y_all.astype(int).reset_index(drop=True)

print("Training feature shape:", X_all.shape)
print("External feature shape:", X_external.shape)
print("Target counts:")
display(y_all.map({0: "No", 1: "Yes"}).value_counts().rename_axis("Class").to_frame("Rows"))
print("Gender categories retained:", sorted(X_all["Gender"].dropna().astype(str).unique().tolist()))
print("Treatment categories after normalization:", sorted(X_all["Treatment Type"].dropna().astype(str).unique().tolist()))


## 4. Feature and target exploration

Inspect the cleaned feature schema, target balance, numeric distributions and selected categorical frequencies. Missing predictor values are intentionally retained here and handled later by the training-only pipeline.


In [ ]:
print("Feature matrix shape:", X_all.shape)
print("External feature matrix shape:", X_external.shape)
print("Missing predictor values before imputation:", int(X_all.isna().sum().sum()))

class_counts = y_all.map({0: "No", 1: "Yes"}).value_counts()
display(class_counts.rename_axis("Survival Prediction").to_frame("Rows"))

ax = class_counts.plot(kind="bar", title="Training target distribution")
ax.set_xlabel("Survival Prediction")
ax.set_ylabel("Rows")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

for column in NUMERIC_FEATURES:
    ax = X_all[column].dropna().plot(
        kind="hist",
        bins=30,
        title=f"Distribution of {column}",
    )
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()

for column in ["Cancer Stage", "Gender", "Treatment Type", "Country"]:
    print(f"Top categories — {column}")
    display(X_all[column].value_counts(dropna=False).head(10).to_frame("Rows"))


## 5. Stratified internal validation

The external test set is not used for model selection or evaluation. A reproducible 80/20 stratified split is made only from the labelled training data. The validation set is held out until evaluation.


In [ ]:
X_train, X_validation, y_train, y_validation = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_all,
)

print("Training fold:", X_train.shape, y_train.value_counts().to_dict())
print("Validation fold:", X_validation.shape, y_validation.value_counts().to_dict())


## 6. Training-only preprocessing and model

Categorical values are imputed before one-hot encoding. Numerical values are imputed before scaling. The preprocessing transformer is inside an imbalanced-learn pipeline, so its learned parameters and ADASYN resampling are fitted only on the training fold during validation.


In [ ]:
categorical_features = [
    column for column in EXPECTED_FEATURES if column not in NUMERIC_FEATURES
]
numerical_features = NUMERIC_FEATURES.copy()

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_features),
        ("numerical", numerical_pipeline, numerical_features),
    ],
    remainder="drop",
)

model_pipeline = ImbalancedPipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("adasyn", ADASYN(random_state=RANDOM_STATE)),
        ("classifier", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]
)


In [ ]:
validation_model = clone(model_pipeline)
validation_model.fit(X_train, y_train)

validation_predictions = validation_model.predict(X_validation)
validation_probabilities = validation_model.predict_proba(X_validation)[:, 1]


In [ ]:
validation_accuracy = accuracy_score(y_validation, validation_predictions)
validation_precision = precision_score(
    y_validation, validation_predictions, zero_division=0
)
validation_recall = recall_score(
    y_validation, validation_predictions, zero_division=0
)
validation_f1 = f1_score(
    y_validation, validation_predictions, zero_division=0
)
validation_balanced_accuracy = balanced_accuracy_score(
    y_validation, validation_predictions
)
validation_roc_auc = roc_auc_score(
    y_validation, validation_probabilities
)

cm = confusion_matrix(y_validation, validation_predictions, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
validation_specificity = tn / (tn + fp) if (tn + fp) else np.nan

metrics = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall (Sensitivity)",
            "F1-score",
            "Specificity",
            "Balanced accuracy",
            "ROC-AUC",
        ],
        "Value": [
            validation_accuracy,
            validation_precision,
            validation_recall,
            validation_f1,
            validation_specificity,
            validation_balanced_accuracy,
            validation_roc_auc,
        ],
    }
)

display(metrics.style.format({"Value": "{:.4f}"}))
print("Confusion matrix (rows = actual [No, Yes], columns = predicted [No, Yes]):")
print(cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No", "Yes"],
).plot(cmap="Blues")
plt.title("Stratified validation confusion matrix")
plt.tight_layout()
plt.show()


## 7. Fit the candidate model on all labelled data and predict the external test set

After the internal validation metrics have been recorded, a fresh copy of the same pipeline is fitted on all labelled training rows. The untouched external test predictors are then scored. The original external IDs are copied into the output in their original row order.

The existing deployed `GradientBoosting.pkl` is not overwritten. A separate candidate artifact is written for later compatibility testing.


In [ ]:
final_model = clone(model_pipeline)
final_model.fit(X_all, y_all)

external_predictions = final_model.predict(X_external)
prediction_labels = pd.Series(external_predictions).map({0: "No", 1: "Yes"})

submission = pd.DataFrame(
    {
        "ID": external_test_ids,
        "Survival Prediction": prediction_labels,
    }
)

if len(submission) != EXTERNAL_TEST_ROWS:
    raise AssertionError(
        f"Expected {EXTERNAL_TEST_ROWS:,} prediction rows; got {len(submission):,}."
    )

if submission.columns.tolist() != ["ID", "Survival Prediction"]:
    raise AssertionError("Submission columns do not match the required schema.")

if not submission["ID"].reset_index(drop=True).equals(external_test_ids):
    raise AssertionError("Original external test IDs or their row order changed.")

if submission["Survival Prediction"].isna().any():
    raise AssertionError("The prediction output contains missing labels.")

submission_path = Path("Sample Prediction File.csv")
submission.to_csv(submission_path, index=False)

candidate_model_path = Path("GradientBoosting_candidate.pkl")
with candidate_model_path.open("wb") as model_file:
    pickle.dump(final_model, model_file)

print(f"Saved submission: {submission_path.resolve()}")
print(f"Submission rows: {len(submission):,}")
print("Submission columns:", submission.columns.tolist())
print(f"Saved candidate model: {candidate_model_path.resolve()}")
print("The deployed GradientBoosting.pkl was not changed.")


In [ ]:
# Final output integrity checks.
saved_submission = pd.read_csv(
    submission_path,
    dtype={ID_COLUMN: external_test_ids.dtype},
)

assert len(saved_submission) == 75_000
assert saved_submission.columns.tolist() == ["ID", "Survival Prediction"]
assert saved_submission["ID"].equals(external_test_ids)
assert set(saved_submission["Survival Prediction"].dropna().unique()).issubset({"Yes", "No"})

display(saved_submission.head())
display(saved_submission.tail())
print("PASS: 75,000 rows, required columns, original IDs and Yes/No labels verified.")


## 8. Interpretation and limitations

- Report the metrics generated in Section 4 only after running this notebook with the actual training data. Do not copy the older, non-stratified scores as if they came from this corrected workflow.
- The external test data has no labels in this workflow, so external accuracy, recall, specificity, balanced accuracy, ROC-AUC and a confusion matrix cannot be calculated from it.
- ADASYN changes the training-fold class distribution; it is not applied to validation or external test rows.
- `P` is retained as an observed gender category because its meaning is undocumented. Confirm its definition with the dataset owner before interpreting subgroup results or adding it to the app.
- The treatment labels are normalized to the current Streamlit options. Test the candidate model against the actual app before deployment.
- This project is an educational exercise. It has not undergone clinical validation, external clinical evaluation, calibration assessment, fairness review, or regulatory review.
